# Calculating Power-law scaling in fluctuations of information

In [1]:
import os
import numpy as np
import pandas as pd
from scipy.stats import t

import pyarrow.parquet as pq
from tqdm.notebook import tqdm
import warnings;warnings.filterwarnings('ignore')

In [2]:
CORPUS_NAME = ''

In [3]:
DATA_PATH = '../../../updated-data/obs/lme-ready'
NULL_DATA_PATH = '../../../updated-data/null/null-lme-ready'

OUTPUTS_PATH = 'reports'
if not os.path.exists(OUTPUTS_PATH):
    os.mkdir(OUTPUTS_PATH)

In [4]:
def grab_all_files(PATH, file_type='.parquet'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('.'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

def lang_id(x):
    if 'xling-' in x:
        return x.split('-')[2]
    else:
        return 'eng'

def corpus_id(x):
    if 'xling-' in x:
        return x.split('-')[1]
    else:
        return x.split('-')[0]

In [5]:
fs = [
    f for f in os.listdir(DATA_PATH)
    if ('xling-' not in f)
    and (not f.startswith('.'))
    and (f.endswith('.parquet'))
    and (CORPUS_NAME in f)
]
len(fs)

4518

## Processing individual datasets

In [6]:
# formula = '{} ~ nx + ny + self'.format(COEF_VAR)

In [7]:
COEF_VAR = 'AVAR'

In [8]:
param_list = 'time_delta'

In [9]:
df_scale, df_regression, k_docs, error_tracker = [], [], dict(), 0

In [10]:
for f in tqdm(fs):
    # print('===][===')
    # print(f.split('/')[-1])

    table = pq.ParquetFile(os.path.join(DATA_PATH, f))

    df = table.read(columns=[
        'corpus', 'conversation_id', 'x_speaker', 'y_speaker',
        COEF_VAR,
        'x_turn_id', 'y_turn_id',
        # 'nx', 'ny'
    ]).to_pandas()
    sel = df['y_turn_id'] - df['x_turn_id']

    df['self'] = (df['x_speaker'] == df['y_speaker'])
    # df['x_turn_id'] = [int(tid.split('-')[-1]) for tid in df['x_turn_id']]

    # find and eliminate bad y_turns
    #   The idea is that some speakers only generate one utterance per
    #   conversation, and thus grouping the way we're doing here yields
    #   errors when calc'ing power-law exponents
    #
    #   (1) .drop dupes on y_speaker, y_turn_id
    #   (2) ["y_speaker"].count_values()
    #   (3) k > 1

    df = df.loc[
        (~df[COEF_VAR].isna())
        & (sel > 0) & (sel <= 200)
    ]

    groups = df.groupby(by=['x_speaker', 'y_speaker'])

    for labels, dfi in groups:
        self_other = dfi['self'].unique()[0]
        dfi['min_y_turn_id'] = dfi.groupby('x_turn_id')['y_turn_id'].transform('min')

        try:
            if len(dfi) > 1:

                b, a = np.polyfit(
                    np.log((dfi['y_turn_id'] - dfi['min_y_turn_id']).values + 1),
                    np.log(dfi['AVAR'].values + 1e-9),
                    1
                )

                df_regression += [
                    {
                        'corpus': corpus_id(f),
                        'cid': f.replace('.parquet', ''),
                        'speaker': labels[0],
                        'self': self_other,
                        'a': np.exp(a),
                        'b': b,
                        'k': len(dfi)
                    }
                ]

        except Exception:
            error_tracker += 1

df_regression = pd.DataFrame(df_regression)

  0%|          | 0/4518 [00:00<?, ?it/s]

** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  5 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  5 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number  4 had an illegal value
** On entry to DLASCL, parameter number 

In [11]:
len(df_regression), error_tracker

(42286, 1697)

In [12]:
# df_regression.isna().sum()

In [13]:
# df_regression = df_regression.loc[
#     (df_regression.isna().astype(int).sum(axis=1) == 0)
#     & (np.isinf(df_regression).astype(int).sum(axis=1) == 0)
# ]

In [14]:
df_regression.to_parquet(
    os.path.join(OUTPUTS_PATH,'all.parquet'),
    compression='snappy', engine='fastparquet'
)

## Corpus level analysis of results

In [15]:
split_by = ['corpus', 'self']

In [16]:
df0 = df_regression[split_by+['b']].groupby(by=split_by).agg('mean').reset_index()
df0['std'] = df_regression[split_by+['b']].groupby(by=split_by).agg('std').reset_index()['b'].values

# df0['k'] = [k_docs[corpus][s] for corpus, s in df0[['corpus', 'self']].values]
df0['k'] = df_regression[split_by+['b']].groupby(by=split_by).agg('count').reset_index()['b'].values

df0['se'] = df0['std'] / np.sqrt(df0['k'])

In [17]:
df0['t'] = df0['b'] / df0['se']
df0['p'] = [t.sf(tstat.__abs__(), df=k-1) for tstat, k in df0[['t', 'k']].values]

In [18]:
df0.head(n=50)

,corpus,self,b,std,k,se,t,p
0,CABNC,False,-0.186180,0.823084,12964,0.007229,-25.754772,5.354141e-143
1,CABNC,True,-0.290896,0.751304,4734,0.010919,-26.640102,4.090544e-146
2,CANDOR,False,-0.388708,0.121550,3306,0.002114,-183.873265,0.000000e+00
3,CANDOR,True,-0.385761,0.130612,3305,0.002272,-169.793451,0.000000e+00
4,CORAAL,False,-0.176010,0.309634,699,0.011711,-15.028923,9.781319e-45
5,CORAAL,True,-0.233393,0.184146,569,0.007720,-30.232968,1.094875e-120
6,DISPEL,False,-0.382432,0.462721,38,0.075063,-5.094807,5.260058e-06
7,DISPEL,True,-0.372080,0.237882,37,0.039108,-9.514285,1.154758e-11
8,Frederiksen,False,-0.307967,0.345341,4,0.172670,-1.783556,8.625033e-02
9,Frederiksen,True,-0.351457,0.419401,4,0.209700,-1.675995,9.616629e-02


In [19]:
df0.to_csv(
    os.path.join(OUTPUTS_PATH, 'corpus-results.csv'),
    index=False, encoding='utf-8'
)

## Aggregate analysis

In [20]:
split_by = ['self']

In [21]:
df0 = df_regression[split_by+['b']].groupby(by=split_by).agg('mean').reset_index()
df0['std'] = df_regression[split_by+['b']].groupby(by=split_by).agg('std').reset_index()['b'].values
df0['k'] = df_regression[split_by+['b']].groupby(by=split_by).agg('count').reset_index()['b'].values

# df0['k'] = [k_docs[corpus][s] for corpus, s in df0[['corpus', 'self']].values]

df0['se'] = df0['std'] / np.sqrt(df0['k'])

In [22]:
df0['t'] = df0['b'] / df0['se']
df0['p'] = [t.sf(tstat.__abs__(), df=k-1) for tstat, k in df0[['t', 'k']].values]

In [23]:
df0.head(n=20)

,self,b,std,k,se,t,p
0,False,-0.168413,0.800822,31446,0.004516,-37.292647,3.391388e-298
1,True,-0.310090,0.599336,10840,0.005756,-53.868117,0.000000e+00


In [24]:
df0.to_csv(
    os.path.join(OUTPUTS_PATH, 'agg-results.csv'),
    index=False,
    encoding='utf-8'
)